# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.85114127  0.70844344  0.76660115  0.30604193  0.13723245]
 [ 0.67436314  0.44853947  0.00625694  0.55995387  0.99853495]
 [ 0.96760288  0.67187338 -0.5606992   0.66235561  0.21824379]
 [ 0.81408241 -0.96979642  0.46042945  0.13088015  0.35182707]
 [ 0.42067482  0.60576274  0.86120632 -0.71349124  0.81349578]
 [-0.22732897  0.14659923  0.22457903  0.49291001 -0.60003375]
 [ 0.62787556 -0.24513342  0.01505053 -0.22830714 -0.45095942]
 [-0.34408694  0.27961318  0.32538775 -0.81230746 -0.69337039]
 [ 0.7378066  -0.44892581 -0.17969183 -0.56365378  0.83519979]
 [-0.59414845  0.1503769  -0.98683619 -0.66672427 -0.98298963]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a2', 'a2', 'a2', 'a1', 'a1', 'a2', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 1, 0, 1, 1, 1, 0, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.03s/it, loss=2208.8557]

SVI:   6%|▌         | 2/34 [00:01<00:33,  1.03s/it, loss=3012.1340]

SVI:   9%|▉         | 3/34 [00:01<00:32,  1.03s/it, loss=2376.6033]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.13it/s, loss=3312.6604]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.13it/s, loss=3179.0957]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.13it/s, loss=2638.6377]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.20it/s, loss=2638.6377]

SVI: 100%|██████████| 34/34 [00:01<00:00, 23.20it/s, loss=1818.1746]